# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

In [1]:
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow trl torchvision torchaudio
!pip install -r requirements.txt


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Set PyTorch CUDA memory config for better allocation
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'


In [3]:
# 1. Imports & Configuration 

import os
import json
import random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import torch.optim as optim
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig, TaskType

# Reproducibility 
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Paths 
# Adjust these paths to match your local environment
DATA_DIR   = Path("data")

# Model 
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# Basic Settings 
IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


/home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
GPU: NVIDIA GeForce RTX 3080


## 2. Load and Preprocess Data

In [4]:
# 2a. Load CSVs 
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

Train: 3,109 | Val: 1,048 | Test: 1,008


,id,image_path,question,choices,num_choices,answer,hint,lecture,solution,task,grade,subject,topic,category,skill
0,train_07667,images/train/train_07667.png,Why might putting each tadpole in its own pool...,[the male's tadpoles will be larger when they ...,3,2,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...
1,train_02628,images/train/train_02628.png,Why might forming strong social bonds with oth...,"[the female's offspring will live longer, the ...",3,0,Animals often behave in certain ways that can ...,Animals increase their reproductive success wh...,Look for the part of the passage that describe...,closed choice,grade8,natural science,literacy-in-science,Adaptations and natural selection,How can animal behaviors affect reproductive s...


In [5]:
# 2b. Prompt Engineering 
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    """
    Builds the text prompt for the Vision Language Model.
    The <image> token is required for the model to process the image.
    """
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices)
    )

    prompt = "<image>\n"
    if context_str:
        prompt += f"Context:\n{context_str}\n\n"
    prompt += f"Question: {row['question']}\n"
    prompt += f"Choices:\n{choices_str}\n"
    prompt += "Answer: "

    if include_answer:
        answer_idx = int(row['answer'])
        prompt += f"{CHOICE_LETTERS[answer_idx]}"

    return prompt

# Display an example prompt
print(build_prompt(train_df.iloc[0], include_answer=True))


<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always successful

In [6]:
# 2c. PyTorch Dataset
import torchvision.transforms as T

class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train
        
        if self.is_train:
            self.transform = T.Compose([
T.Resize((self.img_size, self.img_size), interpolation=T.InterpolationMode.BICUBIC)
            ])
        else:
            self.transform = T.Compose([
                T.Resize((self.img_size, self.img_size), interpolation=T.InterpolationMode.BICUBIC)
            ])

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = self.transform(img)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer":  int(row["answer"]) if "answer" in row else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=True)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")


Datasets created: train=3109, val=1048, test=1008


## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [7]:
# 3a. Load SmolVLM model + run one inference example 
from transformers import AutoProcessor, AutoModelForVision2Seq

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True,
 )
if not torch.cuda.is_available():
    model.to(device)
model.eval()

# Pick a sample from validation set
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Prompt:")
print(sample_prompt)
print("\nModel output:")
print(decoded)
print(f"\nGround-truth answer index: {sample['answer']}")

/home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(


Prompt:
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mates won't always su

## 3.5. QLoRA Training Configuration & Loop

In [ ]:
#  3.5a. QLoRA Setup

# 4-Bit Quantization Config to stay within 15GB VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# LoRA Settings
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=6,
    lora_alpha=16,
    target_modules="all-linear",
    lora_dropout=0.1,
use_dora=False,
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


/home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


trainable params: 4,331,136 || all params: 511,813,440 || trainable%: 0.8462


In [ ]:
# 3.5b. DataLoader Setup for Training 
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
EPOCHS = 2  
NUM_WORKERS = 1  

def train_collate_fn(batch):
    images = [item["image"] for item in batch]
    texts = [item["text"] for item in batch]
    
    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    
    labels = inputs["input_ids"].clone()
    
    # PROMPT MASKING 
    for i in range(labels.shape[0]):
        # Find the length of the actual sentence, ignoring the [PAD] tokens at the end
        seq_len = (inputs["attention_mask"][i] == 1).sum().item()
        # Mask everything (-100) except the last 2 non-pad tokens (e.g. ":", " A")
        labels[i, :seq_len - 2] = -100
        
    # Mask pad tokens so the model does not learn to predict padding
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    return inputs

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, 
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=train_collate_fn
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, 
    num_workers=NUM_WORKERS, pin_memory=True, collate_fn=train_collate_fn
)


In [ ]:
#  3.5c. PyTorch Train Loop with Checkpointing 
optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_dir = f"models/{timestamp}"
os.makedirs(f"{model_dir}/checkpoints", exist_ok=True)
os.makedirs(f"{model_dir}/final", exist_ok=True)

import pandas as pd
metrics_csv = "models/training_metrics.csv"
if not os.path.exists(metrics_csv):
    pd.DataFrame(columns=["Model Name", "Epoch", "Step", "Train Loss", "Val Loss"]).to_csv(metrics_csv, index=False)


print(f"{'Epoch':<8} | {'Step/Total':<15} | {'Train Loss':<12} | {'Val Loss':<12}")
print("-" * 55)

for epoch in range(EPOCHS):
    model.train()
    running_train_loss = 0
    steps_logged = 0
    save_steps = len(train_loader) // 3
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1} Train", leave=False)
    for idx, batch in enumerate(train_pbar):
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        outputs = model(**inputs)
        
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()
        
        running_train_loss += outputs.loss.item()
        steps_logged += 1
        
        if (idx + 1) % GRAD_ACCUM_STEPS == 0 or (idx + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            optimizer.zero_grad()
            
        # Validation Phase triggered at specific checkpoints
        if (idx + 1) % save_steps == 0 or (idx + 1) == len(train_loader):
            avg_train_loss = running_train_loss / steps_logged
            
            model.eval()
            total_val_loss = 0
            with torch.inference_mode():
                for v_batch in tqdm(val_loader, desc=f"Step {idx+1} Val", leave=False):
                    v_inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in v_batch.items()}
                    v_outputs = model(**v_inputs)
                    total_val_loss += v_outputs.loss.item()
            
            avg_val_loss = total_val_loss / len(val_loader)
            model.train() # Flip backend to training mode!
            
            step_str = f"{idx+1}/{len(train_loader)}"
            print(f"{epoch+1:<8} | {step_str:<15} | {avg_train_loss:<12.4f} | {avg_val_loss:<12.4f}")
            
            # Append to persistent CSV Tracker
            new_row = {
                "Model Name": timestamp,
                "Epoch": epoch + 1,
                "Step": idx + 1,
                "Train Loss": round(avg_train_loss, 4),
                "Val Loss": round(avg_val_loss, 4)
            }
            pd.DataFrame([new_row]).to_csv(metrics_csv, mode="a", header=False, index=False)
            
            # Checkpoint the model parameters
            step_save_path = f"{model_dir}/checkpoints/epoch_{epoch+1}_step_{idx+1}"
            model.save_pretrained(step_save_path)
            
            # Reset rolling counters for next third of epoch
            running_train_loss = 0
            steps_logged = 0

# Final Isolated Save
final_save_path = f"{model_dir}/final"
model.save_pretrained(final_save_path)
print(f"\nTraining complete! Final model beautifully saved at: {final_save_path}")

Epoch    | Step/Total      | Train Loss   | Val Loss    
-------------------------------------------------------


Epoch 1 Train:   0%|          | 0/3109 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 1 Train:  33%|███▎      | 1035/3109 [28:08<56:22,  1.63s/it] 

1        | 1036/3109       | 0.3891       | 0.3732      


Epoch 1 Train:  33%|███▎      | 1036/3109 [36:01<82:21:18, 143.02s/it]/home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 1 Train:  67%|██████▋   | 2071/3109 [1:03:42<27:30,  1.59s/it]  

1        | 2072/3109       | 0.3140       | 0.3381      


...

Epoch 1 Train: 100%|█████████▉| 3108/3109 [1:47:11<02:23, 143.42s/it]/home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


1        | 3109/3109       | 0.6405       | 0.3005      


Epoch 2 Train:   0%|          | 0/3109 [00:00<?, ?it/s]              /home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 2 Train:  33%|███▎      | 1035/3109 [27:41<54:49,  1.59s/it] 

2        | 1036/3109       | 0.2245       | 0.3268      


Epoch 2 Train:  33%|███▎      | 1036/3109 [35:38<82:53:01, 143.94s/it]/home/anthonylamelas/Coding/sci_mc_pred/.venv/lib/python3.13/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
Epoch 2 Train:  67%|██████▋   | 2071/3109 [1:03:37<28:30,  1.65s/it]  

## 4. Full Batch Inference (Test Set)

In [ ]:
# 4a. Batch Inference Setup 
BATCH_SIZE = 1
NUM_WORKERS = 1

def collate_fn(batch):
    return {
        "images": [item["image"] for item in batch],
        "texts": [item["text"] for item in batch]
    }

test_loader = DataLoader(
    test_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=True,
    collate_fn=collate_fn
)

# Validation inference loader uses the same prompt format as test
val_inf_ds = ScienceQADataset(val_df, DATA_DIR, img_size=IMG_SIZE, is_train=False)
val_inf_loader = DataLoader(
    val_inf_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=collate_fn
)
,

In [ ]:
# best_checkpoint_path = "models/20260506_113510"
# try:
#     model.load_adapter(best_checkpoint_path, "default")
#     model.set_adapter("default")
#     print(f"Loaded 74% best model checkpoint: {best_checkpoint_path}")
# except Exception as e:
#     print(f"Could not load checkpoint. Error: {e}")


In [ ]:
# 4b. Run Inference 
predictions = []
generated_texts = []  # For debugging

# Fix padding for batch inference
processor.tokenizer.padding_side = 'left'

CHOICE_LETTERS = "ABCDEFGHIJ"

model.eval()
with torch.inference_mode():
    for batch in tqdm(test_loader, desc="Generating Predictions"):
        processor_inputs = processor(
            text=batch["texts"],
            images=batch["images"],
            return_tensors="pt",
            padding=True,
            truncation=True,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in processor_inputs.items()}
        
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            num_beams=1,  
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
        )

        # Extract only the generated tokens 
        generated_ids = outputs.sequences[:, inputs['input_ids'].shape[1]:]
        decoded_sequences = processor.batch_decode(generated_ids, skip_special_tokens=True)
        for decoded in decoded_sequences:
            generated_text = decoded.strip()
            generated_texts.append(generated_text)  # Store for debug
            if not generated_text:
                predictions.append(0)
                continue

            import re
            normalized = generated_text.upper().strip()

            match = re.search(r'ANSWER[:\s]*([A-J])', normalized)
            if match is None:
                match = re.search(r'\b([A-J])\b', normalized)

            if match:
                letter = match.group(1)
                predictions.append(CHOICE_LETTERS.index(letter))
            else:
                fallback = re.sub(r'[^A-J]', '', normalized)
                if fallback:
                    predictions.append(CHOICE_LETTERS.index(fallback[0]))
                else:
                    predictions.append(0)
        
        # Clear cache after each batch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
# 4b. Final Test Inference for All Configs

CHOICE_LETTERS = "ABCDEFGHIJ"
processor.tokenizer.padding_side = 'left'

param_configs = [
    {"max_new_tokens": 2, "num_beams": 1, "temperature": None, "desc": "Greedy-2tok"},
#     {"max_new_tokens": 3, "num_beams": 1, "temperature": None, "desc": "Greedy-3tok"},
#     {"max_new_tokens": 5, "num_beams": 1, "temperature": None, "desc": "Greedy-5tok"},
#     {"max_new_tokens": 2, "num_beams": 2, "temperature": None, "desc": "Beam2-2tok"},
#     {"max_new_tokens": 3, "num_beams": 2, "temperature": None, "desc": "Beam2-3tok"},
#     {"max_new_tokens": 2, "num_beams": 3, "temperature": None, "desc": "Beam3-2tok"},
 ]


def decode_generated_sequences(decoded_sequences):
    import re
    predictions = []
    generated_texts = []
    for decoded in decoded_sequences:
        generated_text = decoded.strip()
        generated_texts.append(generated_text)
        if not generated_text:
            predictions.append(0)
            continue

        normalized = generated_text.upper().strip()
        match = re.search(r'ANSWER[:\s]*([A-J])', normalized)
        if match is None:
            match = re.search(r'\b([A-J])\b', normalized)

        if match:
            predictions.append(CHOICE_LETTERS.index(match.group(1)))
        else:
            fallback = re.sub(r'[^A-J]', '', normalized)
            if fallback:
                predictions.append(CHOICE_LETTERS.index(fallback[0]))
            else:
                predictions.append(0)
    return predictions, generated_texts


def save_submission(preds, config_desc):
    output_dir = Path('submissions')
    output_dir.mkdir(parents=True, exist_ok=True)
    safe_desc = config_desc.replace(' ', '_').replace('-', '').lower()
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    filename = f'submission_{safe_desc}_{timestamp}.csv'
    save_path = output_dir / filename
    pd.DataFrame({'id': test_df['id'], 'answer': preds}).to_csv(save_path, index=False)
    print(f'Saved submission for {config_desc} to: {save_path}')
    return save_path


def run_test_inference(config):
    predictions = []
    generated_texts = []
    print(f"Running test inference for {config['desc']}...")
    model.eval()
    with torch.inference_mode():
        for batch in tqdm(test_loader, desc=f"Test [{config['desc']}]", leave=False):
            processor_inputs = processor(
                text=batch['texts'],
                images=batch['images'],
                return_tensors='pt',
                padding=True,
                truncation=True,
            )
            inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in processor_inputs.items()}

            outputs = model.generate(
                **inputs,
                max_new_tokens=config['max_new_tokens'],
                num_beams=config['num_beams'],
                do_sample=False,
                return_dict_in_generate=True,
            )

            generated_ids = outputs.sequences[:, inputs['input_ids'].shape[1]:]
            decoded_sequences = processor.batch_decode(generated_ids, skip_special_tokens=True)
            batch_preds, batch_texts = decode_generated_sequences(decoded_sequences)
            predictions.extend(batch_preds)
            generated_texts.extend(batch_texts)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    return predictions, generated_texts


results = []

for config in param_configs:
    print(f"\n{'='*60}")
    print(f"Testing: {config['desc']}")
    print(f"  max_new_tokens={config['max_new_tokens']}, num_beams={config['num_beams']}")
    print(f"{'='*60}")

    test_predictions, test_generated_texts = run_test_inference(config)
    save_submission(test_predictions, config['desc'])
    results.append({
        'config': config['desc'],
        'max_new_tokens': config['max_new_tokens'],
        'num_beams': config['num_beams'],
    })

print(f"\n{'='*60}")
print(f"{'='*60}")
for r in results:
    print(f"  {r['config']}: max_new_tokens={r['max_new_tokens']}, num_beams={r['num_beams']}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
